<a href="https://colab.research.google.com/github/lilhawkeye2002-ux/notebooks/blob/main/AudioToText_Whisper.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🗣️ AudioToText with Whisper (🛠 [Whisper by OpenAI](https://github.com/openai/whisper))


## [Step 1] ⚙️ Setup Files Install the required libraries

Click ▶️ button below to install the dependencies for this notebook.

In [ ]:
import subprocess
import sys
import os
from platform import platform
from sys import platform as sys_platform

# --- Check and install ffmpeg ---
status, ffmpeg_version = subprocess.getstatusoutput("ffmpeg -version")

if status != 0:
  if sys_platform == 'linux' and 'ubuntu' in platform().lower():
    print("Installing ffmpeg...")
    !apt update -qq && apt install -y ffmpeg
    print("ffmpeg installed.")
  else:
    print("Install ffmpeg: https://ffmpeg.org/download.html")
    sys.exit(1)
else:
  print(f"ffmpeg version: {ffmpeg_version.split('\n')[0]}")

# --- Install build essentials ---
if sys_platform == 'linux' and 'ubuntu' in platform().lower():
    print("Installing build essential tools...")
    !apt update -qq && apt install -y build-essential cmake

# --- Upgrade pip and install dependencies ---
print("Upgrading pip...")
try:
    !pip install --no-warn-script-location --user --upgrade pip
    print("pip upgraded successfully.")
except Exception as e:
    print(f"Warning upgrading pip: {e}")

print("Installing Python dependencies...")
pip_install_cmd = "pip install --root-user-action=ignore openai-whisper openai pydub cohere ffmpeg-python"

try:
    !{pip_install_cmd}
    import whisper
    import torch
    print("Whisper library successfully imported.")
except Exception as e:
    print(f"Error installing dependencies: {e}")
    sys.exit(1)

# --- Pre-load Model ---
use_model = "large-v2" #@param ["tiny", "base", "small", "medium", "large-v1", "large-v2"]
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Pre-loading {use_model} model on {DEVICE}...")
model = whisper.load_model(use_model, device=DEVICE)
print("Model loaded.")

In [ ]:
# @title ## [Step 2] 📁 Run to your upload your audio files or list them manually.

print("Almost any audio or video file format is [supported](https://gist.github.com/Carleslc/1d6b922c8bf4a7e9627a6970d178b3a6).")

# @title
from google.colab import files

uploaded = files.upload()

uploaded_audio_files_str = ','.join(uploaded.keys())
print(f"Uploaded files: {uploaded_audio_files_str}")

In [ ]:
# @title ## [Step 3] 👂 Transcribe
#**3.** If you manually upload multiple files to the notebook's drive, then
# proceed to list their filenames in the form's input, otherwise proceed as the # script will prioritize any audio files uploaded from the previous step.

# audio_file_input = "your filename(s) separated by a comma"

# **3.1** Run this cell and wait for the transcription to complete.
# You can try other parameters if the result with default parameters does not suit your needs.
# Setting the `language` to the language of source audio file may provide better results than Auto-Detect, but Auto-Detect is usually reliable.
# You can add an optional initial `prompt` to provide context about the audio
# or encourage a specific writing style, see the prompting guide
# https://platform.openai.com/docs/guides/speech-to-text/prompting

# While the transcription does not censor output text, I cannot comment on if the additional prompt guidance is ran through safety filters as of April 16 2026
# If the execution takes too long to complete you can choose a smaller model in `use_model`, with an accuracy tradeoff.

# --- Step 3 Configuration ---

import os, subprocess, sys
import whisper
from whisper.utils import format_timestamp, get_writer, WriteTXT
import numpy as np
try:
  import tensorflow
except ImportError:
  pass
import torch
import math
from openai import OpenAI
from typing import TextIO
import re

# --- Stream Wrapper to force real-time output AND write to file ---
class LiveFileWriter:
    def __init__(self, stream, file_path):
        self.stream = stream
        self.file_path = file_path
    def write(self, data):
        # Print to console
        self.stream.write(data)
        self.stream.flush()
        # If it looks like a transcription segment, append to file
        # Whisper verbose output format: [mm:ss.sss -> mm:ss.sss] text
        if " --> " in data:
            text_only = data.split("] ")[-1].strip()
            if text_only:
                with open(self.file_path, "a") as f:
                    f.write(text_only + "\n")
    def writelines(self, datas):
        for d in datas: self.write(d)
    def __getattr__(self, name):
        return getattr(self.stream, name)

# Check for uploaded files from previous step
if 'uploaded_audio_files_str' in globals() and uploaded_audio_files_str:
  audio_file_input = uploaded_audio_files_str
else:
  audio_file_input = "day6_Commentary.m4a" #@param {type:"string"}

audio_files = [path.strip() for path in audio_file_input.split(',') if path.strip()]
for audio_path in audio_files:
  if not os.path.isfile(audio_path):
    print(f"⚠️ Warning: File not found: {audio_path}")

language = "Auto-Detect" #@param ["Auto-Detect", "English", "Mandarin", "Hindi", "Spanish", "French", "Arabic", "Bengali", "Portuguese", "Russian", "Japanese"]
prompt = "" #@param {type:"string"}
coherence_preference = "More coherence, but may repeat text" #@param ["More coherence, but may repeat text", "Less repetitions, but may have less coherence"]
api_key = '' #@param {type:"string"}

# --- Model Setup & Resource Check ---

if api_key:
  DEVICE = "api"
else:
  if 'model' not in globals():
    print("❌ Error: Model not found. Please run Step 1 (Setup) first.")
    raise RuntimeError("Model not found.")
  DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
  if DEVICE == "cpu" and use_model not in ["tiny", "base", "small"]:
    print(f"⚠️ Warning: You are using the '{use_model}' model on CPU. This will be very slow.")

options = {
    'task': "transcribe",
    'verbose': True,
    'fp16': True if DEVICE == "cuda" else False,
    'best_of': 5,
    'beam_size': 5,
    'temperature': (0.0, 0.2, 0.4, 0.6, 0.8, 1.0),
    'condition_on_previous_text': coherence_preference == "More coherence, but may repeat text",
    'initial_prompt': prompt or None,
    'language': None if language == "Auto-Detect" else language.lower()
}

# --- Execution ---

results = {}
output_dir = "audio_transcription"
os.makedirs(output_dir, exist_ok=True)
original_stdout = sys.stdout

try:
  for audio_path in audio_files:
    if not os.path.exists(audio_path): continue
    output_file_name = os.path.splitext(os.path.basename(audio_path))[0]
    txt_path = os.path.join(output_dir, f"{output_file_name}.txt")

    # Clear file if exists
    with open(txt_path, "w") as f: f.write("")

    print(f"\nProcessing: {audio_path}\n")

    if api_key:
      api_client = OpenAI(api_key=api_key)
      with open(audio_path, "rb") as f:
        result = api_client.audio.transcriptions.create(model="whisper-1", file=f, response_format="verbose_json", language=options['language']).model_dump()
        with open(txt_path, "a") as f_out:
          for segment in result['segments']:
            f_out.write(segment['text'].strip() + "\n")
    else:
      # Redirect stdout to our custom writer to capture live segments
      sys.stdout = LiveFileWriter(original_stdout, txt_path)

      local_options = options.copy()
      local_options.pop('verbose', None)

      result = whisper.transcribe(model, audio_path, verbose=True, **local_options)

      # Reset stdout
      sys.stdout = original_stdout

    results[audio_path] = result
finally:
  sys.stdout = original_stdout

# --- Integrated Step 4: Finalize Other Formats ---

if results:
    output_formats = ["vtt", "srt", "tsv", "json"]
    print("\nFinalizing remaining formats...")
    for audio_path, result in results.items():
      output_file_name = os.path.splitext(os.path.basename(audio_path))[0]
      for fmt in output_formats:
        fix_vtt = fmt == 'vtt' and result['segments'] and result['segments'][0].get('start') == 0
        if fix_vtt: result['segments'][0]['start'] += 1/1000
        writer = get_writer(fmt, output_dir)
        writer(result, output_file_name)
        if fix_vtt: result['segments'][0]['start'] = 0
        print(f"Saved: {os.path.join(output_dir, output_file_name)}.{fmt}")
else:
    print("No transcriptions were generated.")

## [Step 4] 💾 **Manually Save results**

Run this cell to MANUALLY write the transcription as a file output.  This should be done automatically during the previous cell.

After running this cell, results will be available in the **audio_transcription** folder in the formats selected in `output_formats`.

If you don't see that folder, you may need to refresh the Files folder.

Available formats: `txt,vtt,srt,tsv,json`

In [ ]:
# set output folder
output_dir = "audio_transcription"

# set output formats
output_formats = "txt,vtt,srt,tsv,json" #@param ["txt,vtt,srt,tsv,json", "txt,vtt,srt", "txt,vtt", "txt,srt", "txt", "vtt", "srt", "tsv", "json"] {allow-input: true}
output_formats = output_formats.split(',')

from typing import TextIO
import os
from whisper.utils import get_writer, WriteTXT

class WriteText(WriteTXT):
  def write_result(self, result: dict, file: TextIO, **kwargs):
    print(result['text'], file=file, flush=True)

def write_result(result, output_format, output_file_name):
  output_format = output_format.strip()

  # start captions in non-zero timestamp
  fix_vtt = output_format == 'vtt' and result['segments'] and result['segments'][0].get('start') == 0
  if fix_vtt:
    result['segments'][0]['start'] += 1/1000

  # write result in the desired format
  writer = WriteText(output_dir) if output_format == 'txt' else get_writer(output_format, output_dir)
  writer(result, output_file_name)

  if fix_vtt:
    result['segments'][0]['start'] = 0

  output_file_path = os.path.join(output_dir, f"{output_file_name}.{output_format}")
  print(output_file_path)

# save results
print("Writing results...")
os.makedirs(output_dir, exist_ok=True)

if 'results' in globals():
  for audio_path, result in results.items():
    print(end='\n')
    output_file_name = os.path.splitext(os.path.basename(audio_path))[0]
    for output_format in output_formats:
      write_result(result, output_format, output_file_name)
else:
  print("No results found. Please run Step 3 first.")

Writing results...
No results found. Please run Step 3 first.
